# **Compulsory Assignment 1**
## *Machine Learning and Deep Learning (CDSCO2041C)*
*Group: MLS26_CA01*

*Student IDs: 185912, 161989, 160714 & 160363*

*Dataset: eth_dkk.csv & XCSE_NOVO-1.csv*

---

## 1. Asset Selection

Should you invest in cryptocurrency (Ethereum) or in equity (Novo Nordisk stock)?

---

In [ ]:
# Imports used in the notebook

import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import SymLogNorm
from IPython.display import Markdown, display


In [ ]:
# Load dataset
file_path = "Data/eth_dkk.csv" 
file_path_2 = "Data/XCSE_NOVO-1.csv"

# Load sheets
# = pd.read_excel(file_path, sheet_name="Variable_list")

In [ ]:
FIG_W = globals().get("FIG_W", 9)
DPI = globals().get("DPI", 140)

In [ ]:
# Initial Dataset Overview
# Inspect variable descriptions

# Clean column names
variable_list.columns = variable_list.columns.str.strip()

# Relevant columns only
print("Table 1: Variable Descriptions")
variables_table = variable_list.iloc[:, :2]
variables_table.columns = ["Variable", "Description"]
variables_table

In [ ]:
# Visualisation of Emission Patterns Sectors and Year over Year Changes
# Fully self contained helper definitions

# -----------------------------
# Imports
# -----------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# -----------------------------
# Helper functions
# -----------------------------
def set_clean_theme():
    """Apply consistent seaborn and matplotlib styling."""
    sns.set_theme(style="white", context="notebook")
    plt.rcParams["figure.facecolor"] = "white"
    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["axes.grid"] = False

def clean_ax(ax):
    """Apply minimal axis styling."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def sparse_xticks(ax, x_values, step=5):
    """Show every step tick for cleaner time axes."""
    x = np.array(x_values)
    if x.size == 0:
        return
    x = np.unique(x)
    x = np.sort(x)
    ticks = x[::step]
    ax.set_xticks(ticks)

def set_xaxis_start_zero(ax, x_values):
    """Remove left margin so the y axis aligns with x equals 0."""
    x = np.array(x_values, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return
    ax.margins(x=0)
    ax.set_xlim(left=0, right=float(np.max(x)))

def caption_title(text):
    """
    Format report captions with one colon only and no hyphens.
    Uses title case while keeping minor words lowercase except first word.
    Preserves standard abbreviations such as YoY.
    """
    text = str(text).replace(":", " ").replace("-", " ")
    text = " ".join(text.split())

    minor_words = {"and", "of", "with", "for", "by", "to", "in", "on", "at", "from", "over"}
    preserve_map = {
        "yoy": "YoY",
        "uk": "UK",
    }

    words = text.split()
    out = []
    for i, w in enumerate(words):
        wl = w.lower()
        if wl in preserve_map:
            out.append(preserve_map[wl])
        elif i > 0 and wl in minor_words:
            out.append(wl)
        else:
            out.append(wl.capitalize())
    return " ".join(out)

# -----------------------------
# Manual statistics
# -----------------------------
def _mean(x):
    x = list(x)
    if len(x) == 0:
        raise ValueError("Empty input.")
    return sum(x) / len(x)

def _sample_var(x):
    x = list(x)
    n = len(x)
    if n < 2:
        return 0.0
    mx = _mean(x)
    return sum((xi - mx) ** 2 for xi in x) / (n - 1)

def covariance(x, y):
    x = list(x)
    y = list(y)
    if len(x) != len(y):
        raise ValueError("x and y must have same length.")
    n = len(x)
    if n < 2:
        return 0.0
    mx = _mean(x)
    my = _mean(y)
    return sum((x[i] - mx) * (y[i] - my) for i in range(n)) / (n - 1)

def correlation(x, y):
    x = list(x)
    y = list(y)
    if len(x) != len(y):
        raise ValueError("x and y must have same length.")
    vx = _sample_var(x)
    vy = _sample_var(y)
    if vx == 0 or vy == 0:
        return 0.0
    return covariance(x, y) / ((vx ** 0.5) * (vy ** 0.5))

# -----------------------------
# Start of visualisation block
# -----------------------------
set_clean_theme()

BASE_YEAR = 1990
sector_col = SECTOR_COL if "SECTOR_COL" in globals() else "Territorial Emissions Statistics sector"
value_col = "Emissions (MtCO2e)"

# Global chart settings
FIG_W = 9
DPI = 140

C_MAP = sns.color_palette("rocket_r", as_cmap=True)
PAL_10 = sns.color_palette("rocket_r", 10)

CBAR = {"pad": 0.02, "aspect": 30}

# Report numbering helpers
TABLE_START = 11
FIGURE_START = 1

table_no = TABLE_START
figure_no = FIGURE_START

def print_table(title):
    global table_no
    print(f"Table {table_no}: {caption_title(title)}")
    table_no += 1

def print_figure(title):
    global figure_no
    print(f"Figure {figure_no}: {caption_title(title)}")
    figure_no += 1

# Heatmap styling settings
COL_DARK = globals().get("COL_DARK", "0.15")
HEAT = {
    "square": True,
    "linewidths": 0.0,
    "xtick_rotation": 45,
    "ytick_rotation": 0,
    "tick_labelsize": 9,
    "annot_color": COL_DARK,
}

def fig_ax(height):
    """Create a standard figure with consistent width and sizing."""
    fig, ax = plt.subplots(figsize=(FIG_W, height), dpi=DPI, constrained_layout=True)
    return fig, ax

def heatmap_fig_axes(data, *, cbar_ratio=0.06, wspace=0.10):
    """
    Create heat map axes with a fixed width and a dedicated colorbar column.
    Figure height is scaled to preserve square cells.
    """
    nrows, ncols = data.shape
    heat_ax_w = FIG_W * (1.0 / (1.0 + cbar_ratio))
    fig_h = heat_ax_w * (nrows / ncols)

    fig = plt.figure(figsize=(FIG_W, fig_h), dpi=DPI)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, cbar_ratio], wspace=wspace)
    ax = fig.add_subplot(gs[0, 0])
    cax = fig.add_subplot(gs[0, 1])
    return fig, ax, cax

def heatmap_fixed(
    data, title, *, cmap=C_MAP, vmin=None, vmax=None, center=None,
    fmt=".2f", cbar_label="", annot=True, annot_size=8
):
    """Draw a heat map with consistent geometry and colorbar formatting."""
    fig, ax, cax = heatmap_fig_axes(data)

    hm = sns.heatmap(
        data.astype(float),
        ax=ax,
        cbar_ax=cax,
        cmap=cmap,
        vmin=vmin, vmax=vmax, center=center,
        annot=annot, fmt=fmt,
        annot_kws={"size": annot_size, "color": "white"},
        square=HEAT["square"],
        linewidths=HEAT["linewidths"],
        cbar_kws={**CBAR, "label": cbar_label},
    )

    ax.set_title(caption_title(title), pad=10)
    ax.tick_params(axis="x", rotation=HEAT["xtick_rotation"], labelsize=HEAT["tick_labelsize"])
    ax.tick_params(axis="y", rotation=HEAT["ytick_rotation"], labelsize=HEAT["tick_labelsize"])
    ax.set_xlabel("")
    ax.set_ylabel("")
    clean_ax(ax)

    cbar = hm.collections[0].colorbar
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label(cbar_label, fontsize=10)

    return fig, ax

# -----------------------------
# Data preparation
# -----------------------------
if "territorial" not in globals():
    raise NameError("Expected a DataFrame named 'territorial' to exist before this cell runs.")

territorial = territorial.copy()
territorial = territorial[territorial["Year"] >= BASE_YEAR]

# Set baseline year index so 1990 equals 0
territorial["Year_idx"] = territorial["Year"] - BASE_YEAR

# Sector pivot table
if "pivot_sec" not in globals():
    pivot_sec = (
        territorial.groupby(["Year", sector_col])[value_col]
        .sum()
        .reset_index()
        .pivot(index="Year", columns=sector_col, values=value_col)
        .fillna(0.0)
        .sort_index()
    )

sectors = pivot_sec.columns.tolist()

# Sector colour palette
SECTOR_PALETTE = sns.color_palette("magma", n_colors=len(sectors))

# First differences for year over year changes
diff_sec = pivot_sec.diff().dropna()

# -----------------------------
# Fuel correlation matrix
# -----------------------------
if "fuel_corr" not in globals():
    FUEL_COL = "Fuel group"
    if FUEL_COL in territorial.columns:
        fuel_year = (
            territorial.dropna(subset=[FUEL_COL])
            .groupby(["Year", FUEL_COL], as_index=False)[value_col]
            .sum()
        )
        pivot_fuel = (
            fuel_year.pivot(index="Year", columns=FUEL_COL, values=value_col)
            .fillna(0.0)
            .sort_index()
        )
        fuel_groups = pivot_fuel.columns.tolist()
        fuel_corr = pd.DataFrame(index=fuel_groups, columns=fuel_groups, dtype=float)
        for g1 in fuel_groups:
            for g2 in fuel_groups:
                fuel_corr.loc[g1, g2] = correlation(pivot_fuel[g1].to_numpy(), pivot_fuel[g2].to_numpy())
    else:
        fuel_corr = pd.DataFrame()

# -----------------------------
# Sector covariance and correlation matrices
# -----------------------------
cov_matrix = pd.DataFrame(index=sectors, columns=sectors, dtype=float)
corr_matrix = pd.DataFrame(index=sectors, columns=sectors, dtype=float)
for s1 in sectors:
    for s2 in sectors:
        cov_matrix.loc[s1, s2] = covariance(pivot_sec[s1].to_numpy(), pivot_sec[s2].to_numpy())
        corr_matrix.loc[s1, s2] = correlation(pivot_sec[s1].to_numpy(), pivot_sec[s2].to_numpy())

diff_corr_matrix = pd.DataFrame(index=sectors, columns=sectors, dtype=float)
for s1 in sectors:
    for s2 in sectors:
        diff_corr_matrix.loc[s1, s2] = correlation(diff_sec[s1].to_numpy(), diff_sec[s2].to_numpy())

# -----------------------------
# Tables
# -----------------------------
sec_change = pd.DataFrame({"1990": pivot_sec.iloc[0], "2024": pivot_sec.iloc[-1]})
sec_change["pct_change_1990_2024"] = (sec_change["2024"] - sec_change["1990"]) / sec_change["1990"].abs() * 100
print_table("Sector Emissions and Percent Change from 1990 to 2024")
display(sec_change.round(2))

years_levels = pivot_sec.index.to_numpy()
years_diff = diff_sec.index.to_numpy()
trend_levels = pd.Series({s: correlation(years_levels, pivot_sec[s].to_numpy()) for s in sectors})
trend_diff = pd.Series({s: correlation(years_diff, diff_sec[s].to_numpy()) for s in sectors})
trend_df = pd.DataFrame({"corr_year_levels": trend_levels, "corr_year_yoy": trend_diff})
trend_df = trend_df.loc[trend_levels.sort_values().index]
print_table("Trend Strength by Sector with Year Correlation")
display(trend_df.round(3))

diff_stats = diff_sec.agg(["mean", "std", "min", "max"]).T
print_table("Year over Year Changes by Sector Summary")
display(diff_stats.round(2))

# -----------------------------
# Figures
# -----------------------------

FIG_1_TITLE = "UK Territorial Emissions over Time"
FIG_2_TITLE = "Sector Emissions over Time"
FIG_3_TITLE = "YoY Change Histograms"
FIG_4_TITLE = "Sector YoY Change Boxplot"
FIG_5_TITLE = "Sector Composition over Time"
FIG_6_TITLE = "Sector Covariance Matrix"
FIG_7_TITLE = "Fuel Group Correlation Matrix"
FIG_8_TITLE = "Sector Correlation Matrix"
FIG_9_TITLE = "Sector YoY Correlation Matrix"
FIG_10_TITLE = "Subsector YoY Scatter Matrix"

# Figure 1
print_figure(FIG_1_TITLE)
tot = territorial.groupby(["Year", "Year_idx"])[value_col].sum().reset_index().sort_values("Year")
fig, ax = fig_ax(3.6)
sns.lineplot(data=tot, x="Year_idx", y=value_col, ax=ax, color=PAL_10[7], linewidth=2.6)
ax.set_title(caption_title(FIG_1_TITLE), pad=10)
ax.set_xlabel(f"Year Index {BASE_YEAR} = 0")
ax.set_ylabel("Emissions (MtCO2e)")
sparse_xticks(ax, tot["Year_idx"].to_numpy(), step=5)
set_xaxis_start_zero(ax, tot["Year_idx"].to_numpy())
clean_ax(ax)
plt.show()

# Figure 2
print_figure(FIG_2_TITLE)
sec_long = (
    territorial.groupby([sector_col, "Year", "Year_idx"])[value_col]
    .sum()
    .reset_index()
    .sort_values([sector_col, "Year"])
)
fig, ax = fig_ax(4.6)
sns.lineplot(
    data=sec_long, x="Year_idx", y=value_col, hue=sector_col,
    palette=SECTOR_PALETTE, linewidth=2.0, ax=ax
)
ax.set_title(caption_title(FIG_2_TITLE), pad=10)
ax.set_xlabel(f"Year Index {BASE_YEAR} = 0")
ax.set_ylabel("Emissions (MtCO2e)")
sparse_xticks(ax, np.sort(sec_long["Year_idx"].unique()), step=5)
set_xaxis_start_zero(ax, np.sort(sec_long["Year_idx"].unique()))
clean_ax(ax)
ax.legend(frameon=False, ncol=2, bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=9, title=None)
plt.show()

# Figure 3
print_figure(FIG_3_TITLE)
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 3.6), dpi=DPI, constrained_layout=True)
fig.suptitle(caption_title(FIG_3_TITLE), y=1.02)

sns.histplot(
    diff_sec["Electricity supply"],
    bins=15, kde=True,
    ax=axes[0],
    alpha=0.45,
    color=PAL_10[7],
    edgecolor=None
)
axes[0].set_title("Electricity Supply", pad=8)
axes[0].set_xlabel("YoY Change in MtCO2e")
axes[0].set_ylabel("Count")
clean_ax(axes[0])

sns.histplot(
    diff_sec["Agriculture"],
    bins=15, kde=True,
    ax=axes[1],
    alpha=0.45,
    color=PAL_10[4],
    edgecolor=None
)
axes[1].set_title("Agriculture", pad=8)
axes[1].set_xlabel("YoY Change in MtCO2e")
axes[1].set_ylabel("Count")
clean_ax(axes[1])

plt.show()

# Figure 4
print_figure(FIG_4_TITLE)
diff_long = diff_sec.reset_index().melt(id_vars=["Year"], var_name="Sector", value_name="YoY_change")
fig, ax = fig_ax(4.8)
sns.boxplot(
    data=diff_long, x="Sector", y="YoY_change",
    order=sectors, hue="Sector", palette=SECTOR_PALETTE,
    dodge=False, width=0.6, fliersize=2.5, linewidth=1.0,
    ax=ax, legend=False
)
for artist in ax.artists:
    artist.set_alpha(0.40)
ax.set_title(caption_title(FIG_4_TITLE), pad=10)
ax.set_xlabel("")
ax.set_ylabel("YoY Change in MtCO2e")
ax.tick_params(axis="x", rotation=90, labelsize=9)
clean_ax(ax)
plt.show()

# Figure 5
print_figure(FIG_5_TITLE)
t = (pivot_sec.index.to_numpy() - pivot_sec.index.min())
year0 = int(pivot_sec.index.min())

fig, ax = fig_ax(4.8)
ax.stackplot(
    t,
    np.vstack([pivot_sec[c].to_numpy() for c in sectors]),
    labels=sectors,
    alpha=0.75,
    colors=[SECTOR_PALETTE[i % len(SECTOR_PALETTE)] for i in range(len(sectors))]
)
ax.set_title(caption_title(FIG_5_TITLE), pad=10)
ax.set_xlabel(f"Year Index {year0} = 0")
ax.set_ylabel("Emissions (MtCO2e)")
sparse_xticks(ax, t, step=5)
set_xaxis_start_zero(ax, t)
clean_ax(ax)
ax.legend(frameon=False, ncol=2, bbox_to_anchor=(1.02, 1.0), loc="upper left", fontsize=9, title=None)
plt.show()

# Heat maps shown last for consistent presentation width

print_figure(FIG_6_TITLE)
cov_display = cov_matrix.clip(lower=0)
heatmap_fixed(
    cov_display,
    FIG_6_TITLE,
    cmap=C_MAP,
    vmin=0, vmax=2600,
    fmt=".0f",
    cbar_label=r"Covariance (MtCO$_2$e)$^2$",
    annot=True,
    annot_size=7
)
plt.show()

print_figure(FIG_7_TITLE)
if isinstance(fuel_corr, pd.DataFrame) and fuel_corr.shape[0] > 0:
    heatmap_fixed(
        fuel_corr,
        FIG_7_TITLE,
        cmap=C_MAP,
        vmin=-1, vmax=1, center=0,
        fmt=".2f",
        cbar_label="Correlation",
        annot=True,
        annot_size=8
    )
    plt.show()
else:
    print("Fuel group correlation matrix is not available because the Fuel group column is missing.")

print_figure(FIG_8_TITLE)
heatmap_fixed(
    corr_matrix,
    FIG_8_TITLE,
    cmap=C_MAP,
    vmin=-1, vmax=1, center=0,
    fmt=".2f",
    cbar_label="Correlation",
    annot=True,
    annot_size=8
)
plt.show()

print_figure(FIG_9_TITLE)
heatmap_fixed(
    diff_corr_matrix,
    FIG_9_TITLE,
    cmap=C_MAP,
    vmin=-1, vmax=1, center=0,
    fmt=".2f",
    cbar_label="Correlation",
    annot=True,
    annot_size=8
)
plt.show()

# -----------------------------
# Subsector scatterplot matrix for year over year changes
# Top four subsectors by total emissions
# -----------------------------
print_figure(FIG_10_TITLE)

set_clean_theme()

SUB_COL = "Territorial Emissions Statistics subsector"
YEAR_COL = "Year"
VAL_COL = value_col

df = uk_by_source.copy()
df.columns = df.columns.astype(str).str.strip()

# Identify the four largest subsectors based on pooled emissions
subs = (
    df.groupby(SUB_COL)[VAL_COL]
      .sum()
      .sort_values(ascending=False)
      .head(4)
      .index
      .tolist()
)

# Build subsector year totals and first differences
sub_year = (
    df[df[SUB_COL].isin(subs)]
    .groupby([YEAR_COL, SUB_COL], as_index=False)[VAL_COL]
    .sum()
)

pivot_sub = (
    sub_year.pivot(index=YEAR_COL, columns=SUB_COL, values=VAL_COL)
            .sort_index()
            .fillna(0.0)
)

diff_sub = pivot_sub.diff().dropna()
diff_slog = np.sign(diff_sub) * np.log1p(np.abs(diff_sub))

# Use a paired sample of years across all panels
N_YEARS = 100
rng = np.random.default_rng(42)
years = diff_slog.index.to_numpy()

if len(years) > N_YEARS:
    keep = np.sort(rng.choice(years, size=N_YEARS, replace=False))
    D = diff_slog.loc[keep, subs].copy()
else:
    D = diff_slog[subs].copy()

# Figure layout
k = len(subs)
fig, axes = plt.subplots(
    k, k,
    figsize=(FIG_W, FIG_W),
    dpi=DPI,
    sharex=True,
    sharey=True
)
fig.subplots_adjust(top=0.90, wspace=0.10, hspace=0.10)

# Consistent bins for diagonal histograms
vals = D.to_numpy().ravel()
vals = vals[np.isfinite(vals)]
bins = np.linspace(np.percentile(vals, 1), np.percentile(vals, 99), 14)

# Plot styling
INK = PAL_10[8]
POINT_ALPHA = 0.55
POINT_SIZE = 18

for i, r in enumerate(subs):
    for j, c in enumerate(subs):
        ax = axes[i, j]

        if i == j:
            ax.hist(D[r].dropna().to_numpy(), bins=bins, alpha=0.90, color=INK)
        else:
            x = D[c].to_numpy()
            y = D[r].to_numpy()
            m = np.isfinite(x) & np.isfinite(y)
            x, y = x[m], y[m]

            ax.scatter(x, y, s=POINT_SIZE, alpha=POINT_ALPHA, color=INK, edgecolors="none")

            if len(x) >= 3:
                rxy = np.corrcoef(x, y)[0, 1]
                ax.text(
                    0.04, 0.92, f"r={rxy:.2f}",
                    transform=ax.transAxes,
                    ha="left", va="top",
                    fontsize=9,
                    color=INK
                )

        ax.grid(False)
        clean_ax(ax)

        if i == k - 1:
            ax.set_xlabel(c, fontsize=9, rotation=35, ha="right")
        else:
            ax.set_xlabel("")
        if j == 0:
            ax.set_ylabel(r, fontsize=9)
        else:
            ax.set_ylabel("")

fig.suptitle(
    f"{caption_title(FIG_10_TITLE)}\nSigned Log Delta Emissions Sample",
    y=0.975
)

plt.show()

---
# 2. Capital Allocation Strategy 

How should the DKK 250,000 be allocated? Should you adopt a pure 
strategy (100% allocation to one asset) or a hybrid strategy (e.g., X% in cryptocurrency and Y% in stock)?
What is the appropriate investment horizon (short-term vs. long-term; months vs. years)?

---

---
# 3. Market Timing and Seasonality 

Are there identifiable seasonal patterns or cyclical trends in cryp-tocurrency and stock returns?
For example, are there specific months or periods that historically
yield higher average returns, suggesting a more favorable entry point?

---